# Notebook 5 — Movement Prediction & Bot Archetypes

**Dataset:** 741 battles, 51 bots, ~48M tick rows (50-bot rumble, 35 rounds/battle)

**Questions:**
1. Can we predict opponent position N ticks ahead? (linear vs. circular vs. ML)
2. Do movement patterns differ by bot archetype? (clustering bots by behavior)
3. Are movement patterns periodic/predictable? (autocorrelation analysis)
4. Does opponent movement change across rounds? (adaptation detection)

**Key concepts:**
- **Lateral velocity** = opponent speed perpendicular to the line between the two robots. This is what makes a robot hard to hit — the faster they move sideways, the more a bullet needs to lead.
- **Heading delta** = how fast the opponent is turning (radians per tick). Combined with velocity, this predicts whether they'll continue straight or curve.
- **Autocorrelation** = does a signal repeat itself? If `lateral_velocity` at time t correlates with `lateral_velocity` at time t+20, the opponent has a ~20-tick movement pattern.
- **Archetype** = a cluster of bots that move similarly. Wave surfers, oscillators, rammers, etc.

In [1]:
# Stratified per-robot sampling — full dataset is ~20 GB, can't fit in RAM.
import sys; sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from _loader import (build_robot_index, load_stratified, attach_opponent_bot,
                     numeric_feature_cols, drop_redundant_features,
                     CSV_ROOT_DEFAULT)

sns.set_theme(style='whitegrid', palette='muted')
CSV_ROOT = CSV_ROOT_DEFAULT

selection = build_robot_index(max_robots=50, battles_per_robot=3, seed=42)
print(f"{len(selection)} (battle, observer-bot) pairs selected.")

Indexed 3888 ticks.csv files across 50 distinct robots from 1 root(s).
Selected 50 robots × ~3 battles = 150 (battle, robot) pairs to load.
150 (battle, observer-bot) pairs selected.


## 1. Data Loading

48M rows won't fit in memory at once. Strategy:
- Load a **stratified sample**: pick 5 random battles per bot (as the observed bot), giving ~50 × 5 = 250 perspectives
- For autocorrelation and adaptation analysis, load full battles for selected bots
- Each ticks.csv has ~30K rows (35 rounds × ~800 ticks/round)

In [2]:
# Show how many distinct observer bots ended up in the selection.
observer_counts = {}
for battle_id, bot in selection:
    observer_counts[bot] = observer_counts.get(bot, 0) + 1
print(f'Bots in sample: {len(observer_counts)}')
for name in sorted(observer_counts):
    print(f'  {name}: {observer_counts[name]} battles')


Bots in sample: 50
  Ali 0.4.9: 3 battles
  Ascendant 1.2.27: 3 battles
  BeepBoop 2.0: 3 battles
  BlackBox 0.0.2: 3 battles
  CHCl3 1.4.2: 3 battles
  Cardigan 1.09: 3 battles
  CassiusClay 2rho.02no: 3 battles
  Chalk 2.6.Be: 3 battles
  Combat 3.25.0: 3 battles
  CunobelinDC 1.2: 3 battles
  Cyanide 1.90: 3 battles
  Diamond 1.8.22: 3 battles
  Domogled 1.2: 3 battles
  Dookious 1.573c: 3 battles
  DrussGT 3.1.7: 3 battles
  Engineer 0.5.4: 3 battles
  Firebird 0.25: 3 battles
  Firestarter 2.0f: 3 battles
  Foilist 1.3.1: 3 battles
  Garm 0.9u: 3 battles
  Gilgalad 1.99.5c: 3 battles
  GresSuffurd 0.4.13: 3 battles
  Holden 1.13a: 3 battles
  Horizon 1.2.2: 3 battles
  Hydra 0.21: 3 battles
  Knight 0.6.28: 3 battles
  Midboss 1q.fast: 3 battles
  Nene 1.0.5: 3 battles
  Neuromancer 7.12: 3 battles
  Phoenix 1.02: 3 battles
  PowerHouse 1.7e3: 3 battles
  Pris 0.92: 3 battles
  PulsarMax 0.8.9: 3 battles
  Raven 3.56j8: 3 battles
  Roborio 1.2.4: 3 battles
  RougeDC willow: 3 batt

In [3]:
# Load sampled ticks via the shared loader (handles multi-root, downcasts to
# float32, adds the `robot_name` column from the directory name).
ticks = load_stratified('ticks.csv', selection, row_frac=0.20)

# Filter to scan ticks (rows where the radar saw the opponent — the rest have
# NaN movement features and would just create gaps).
ticks = ticks[ticks['scan_available'] == 1].copy()

# `robot_name` is the observer in this perspective. Recover the opponent name
# from the sibling directory in the battle folder.
ticks = attach_opponent_bot(ticks, selection, csv_root=CSV_ROOT)
ticks = ticks.dropna(subset=['opponent_bot']).reset_index(drop=True)

# Alias for legacy cells below.
ticks['observer_bot'] = ticks['robot_name']

print(f"Loaded {len(ticks):,} scan ticks from "
      f"{ticks.groupby(['battle_id', 'robot_name']).ngroups} perspectives")
print(f"Memory: {ticks.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Unique observers: {ticks['observer_bot'].nunique()} | "
      f"opponents: {ticks['opponent_bot'].nunique()}")


Loaded 150 ticks.csv files → 1,003,073 rows × 65 cols, 50 robots (~354.6 MB)


Loaded 821,691 scan ticks from 148 perspectives


Memory: 421.3 MB
Unique observers: 50 | opponents: 46


## 2. Bot Movement Archetypes (Question 2)

For each bot, compute summary statistics of their movement behavior across all sampled battles.
Then cluster bots by these summaries.

**Movement features per bot:**
- Mean and std of `opponent_lateral_velocity` (how they move sideways)
- Mean and std of `opponent_velocity` (overall speed)
- Mean and std of `opponent_heading_delta` (turning rate)
- Mean `opponent_time_since_direction_change` (how often they reverse)
- Fraction of ticks where `opponent_is_decelerating` = 1
- Mean `opponent_dist_to_wall_min` (wall proximity)

These capture the **observable** movement personality of each bot.

In [4]:
# Quick sanity check: how many ticks per observer/opponent pair did we load?
pair_counts = (ticks.groupby(['observer_bot', 'opponent_bot'])
               .size().reset_index(name='ticks'))
print(f"Distinct (observer, opponent) pairs in sample: {len(pair_counts)}")
print("\nTicks per opponent (top 10):")
print(ticks['opponent_bot'].value_counts().head(10))


Distinct (observer, opponent) pairs in sample: 146

Ticks per opponent (top 10):
opponent_bot
WhiteFang 2.8.1       54827
Tomcat 3.68           54235
BeepBoop 2.0          40998
Neuromancer 7.12      40919
DrussGT 3.1.7         36582
Diamond 1.8.22        35209
GresSuffurd 0.4.13    34873
Shadow 3.83c          26738
Firebird 0.25         25429
Raven 3.56j8          22264
Name: count, dtype: int64


In [5]:
# Build movement profiles: for each bot, aggregate how it LOOKS as an opponent
profiles = ticks.groupby('opponent_bot').agg(
    lat_vel_mean=('opponent_lateral_velocity', 'mean'),
    lat_vel_std=('opponent_lateral_velocity', 'std'),
    vel_mean=('opponent_velocity', 'mean'),
    vel_std=('opponent_velocity', 'std'),
    vel_abs_mean=('opponent_velocity', lambda x: x.abs().mean()),
    heading_delta_mean=('opponent_heading_delta', lambda x: x.abs().mean()),
    heading_delta_std=('opponent_heading_delta', 'std'),
    dir_change_time_mean=('opponent_time_since_direction_change', 'mean'),
    decel_frac=('opponent_is_decelerating', 'mean'),
    wall_dist_mean=('opponent_dist_to_wall_min', 'mean'),
    angular_vel_abs_mean=('opponent_angular_velocity', lambda x: x.abs().mean()),
    distance_mean=('distance', 'mean'),
    n_ticks=('distance', 'count'),
).reset_index()

print(f'Movement profiles for {len(profiles)} bots')
profiles.sort_values('lat_vel_std', ascending=False).head(10)

Movement profiles for 46 bots

,opponent_bot,lat_vel_mean,lat_vel_std,vel_mean,vel_std,vel_abs_mean,heading_delta_mean,heading_delta_std,dir_change_time_mean,decel_frac,wall_dist_mean,angular_vel_abs_mean,distance_mean,n_ticks
14,Firebird 0.25,-0.114271,6.315326,0.317493,6.428091,5.824489,0.034074,0.052571,200.333160,0.167014,125.437218,0.014014,424.121674,25429
16,Foilist 1.3.1,0.020564,6.144620,-0.115705,6.511263,5.866500,0.033103,0.049976,146.737564,0.150560,65.031525,0.011229,500.954041,8661
27,Phoenix 1.02,-0.006263,6.082114,0.101795,6.212843,5.527257,0.034145,0.053778,83.485443,0.187515,137.467072,0.012351,449.376068,12191
30,PulsarMax 0.8.9,0.048006,6.068273,-0.020941,6.237822,5.588684,0.035498,0.057581,65.073051,0.187410,95.804649,0.011703,469.788727,9007
3,BlackBox 0.0.2,-0.201591,6.017701,0.043159,6.088878,5.414209,0.038236,0.059207,54.449566,0.214217,150.574310,0.013159,424.465729,4164
20,Holden 1.13a,0.173503,6.017635,0.087549,6.214371,5.456190,0.040745,0.059247,67.887215,0.188464,152.843048,0.016241,338.993896,8166
37,Toad 0.14t,-0.052000,5.995881,-0.276086,6.460721,5.848382,0.045086,0.066671,85.712402,0.179468,99.504890,0.010852,504.020691,9322
22,Hydra 0.21,-0.092608,5.957904,0.097727,6.244884,5.346337,0.048845,0.072081,198.260925,0.157879,130.794540,0.014802,387.216705,13789
8,CunobelinDC 1.2,-0.078277,5.954331,-0.066701,6.309396,5.682382,0.041836,0.060646,67.517387,0.185999,75.373528,0.011291,486.413422,8742
42,X2 0.17,-0.024616,5.925692,0.104621,6.491829,5.861059,0.043057,0.064422,129.493347,0.160805,86.848076,0.011986,448.949890,13165


In [6]:
# Cluster bots by movement profile
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

feature_cols = [c for c in profiles.columns if c not in ('opponent_bot', 'n_ticks')]
X = profiles[feature_cols].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Try K=3,4,5 clusters
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, k in enumerate([3, 4, 5]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    ax = axes[i]
    for c in range(k):
        mask = labels == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {c}', alpha=0.7, s=60)
        for j in np.where(mask)[0]:
            ax.annotate(profiles.iloc[j]['opponent_bot'], 
                       (X_pca[j, 0], X_pca[j, 1]),
                       fontsize=6, alpha=0.8)
    ax.set_title(f'K={k} clusters')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.suptitle('Bot Movement Archetypes (PCA + K-Means)', y=1.02, fontsize=14)
plt.show()

In [7]:
# Best K by silhouette score
from sklearn.metrics import silhouette_score

scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    scores.append((k, s))
    print(f'K={k}: silhouette={s:.3f}')

best_k = max(scores, key=lambda x: x[1])[0]
print(f'\nBest K: {best_k}')

# Final clustering
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
profiles['cluster'] = km_final.fit_predict(X_scaled)

# Show cluster members
for c in range(best_k):
    members = profiles[profiles['cluster'] == c]['opponent_bot'].tolist()
    print(f'\nCluster {c} ({len(members)} bots): {", ".join(members)}')

K=2: silhouette=0.584
K=3: silhouette=0.160
K=4: silhouette=0.151
K=5: silhouette=0.157
K=6: silhouette=0.185


K=7: silhouette=0.201

Best K: 2

Cluster 0 (45 bots): Ali 0.4.9, Ascendant 1.2.27, BeepBoop 2.0, BlackBox 0.0.2, CHCl3 1.4.2, Cardigan 1.09, CassiusClay 2rho.02no, Chalk 2.6.Be, CunobelinDC 1.2, Diamond 1.8.22, Domogled 1.2, Dookious 1.573c, DrussGT 3.1.7, Engineer 0.5.4, Firebird 0.25, Firestarter 2.0f, Foilist 1.3.1, Garm 0.9u, Gilgalad 1.99.5c, GresSuffurd 0.4.13, Holden 1.13a, Horizon 1.2.2, Hydra 0.21, Knight 0.6.28, Midboss 1q.fast, Nene 1.0.5, Neuromancer 7.12, Phoenix 1.02, PowerHouse 1.7e3, Pris 0.92, PulsarMax 0.8.9, Raven 3.56j8, Roborio 1.2.4, RougeDC willow, Saguaro 1.0, Seraphim 2.3.1, Shadow 3.83c, Toad 0.14t, Tomcat 3.68, WaveSerpent 2.11, WhiteFang 2.8.1, Wintermute 0.8, X2 0.17, YersiniaPestis 3.0, deBroglie rev0108

Cluster 1 (1 bots): XanderCat 12.9


In [8]:
# Cluster profile comparison — what makes each cluster different?
cluster_summary = profiles.groupby('cluster')[feature_cols].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
key_features = ['lat_vel_std', 'vel_abs_mean', 'heading_delta_mean', 
                'dir_change_time_mean', 'decel_frac', 'wall_dist_mean']
titles = ['Lateral Vel Std\n(movement unpredictability)', 
          'Abs Velocity Mean\n(overall speed)',
          'Heading Delta Mean\n(turning intensity)',
          'Time Since Dir Change\n(reversal frequency)',
          'Deceleration Fraction\n(stop-and-go tendency)',
          'Wall Distance Mean\n(wall proximity)']

for ax, feat, title in zip(axes.flat, key_features, titles):
    profiles.boxplot(column=feat, by='cluster', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Cluster')

plt.suptitle('Movement Profile by Cluster', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Movement Periodicity & Predictability (Question 3)

**Autocorrelation** measures whether a signal repeats itself at a fixed lag.
If `opponent_lateral_velocity` at tick t is correlated with the value at tick t+20,
the opponent has a ~20-tick oscillation pattern.

- High autocorrelation at regular lags = **oscillator** (predictable, exploitable)
- Low autocorrelation everywhere = **random mover** (hard to predict)
- Decaying autocorrelation = **smooth mover** (somewhat predictable short-term)

> **CORRECTED (nb11):** The original version of this cell pooled all ticks for a bot
> across all battles/rounds into one vector, destroying temporal structure at round
> boundaries. The corrected version below computes autocorrelation **per-round** and
> averages, giving the true within-round signal.

In [9]:
# Autocorrelation analysis for selected bots — CORRECTED (per-round, not pooled)
# The original version pooled all ticks across battles/rounds into one vector,
# destroying temporal structure at round boundaries → near-zero autocorrelation.
# This version groups by (battle_id, round, robot_name), computes autocorrelation
# within each round, and averages across rounds.

analysis_bots = ['BeepBoop 2.0', 'Diamond 1.8.22', 'DrussGT 3.1.7', 
                 'Shadow 3.83c', 'ScalarR 0.005h.053-noshield',
                 'Saguaro 1.0', 'Firestarter 2.0f', 'Dookious 1.573c']
analysis_bots = [b for b in analysis_bots if b in ticks['opponent_bot'].unique()]

max_lag = 60

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, bot_name in zip(axes.flat, analysis_bots):
    bot_ticks = ticks[ticks['opponent_bot'] == bot_name]
    
    # Compute per-round autocorrelation and average
    acorr_sums = np.zeros(max_lag)
    n_rounds = 0
    for (bid, rnd, robot), grp in bot_ticks.groupby(['battle_id', 'round', 'robot_name']):
        grp = grp.sort_values('tick')
        lat = grp['opponent_lateral_velocity'].values.astype(float)
        if len(lat) < max_lag + 10 or np.nanvar(lat) < 1e-6:
            continue
        for lag in range(1, max_lag + 1):
            valid = ~np.isnan(lat[:-lag]) & ~np.isnan(lat[lag:])
            if valid.sum() > 5:
                acorr_sums[lag - 1] += np.corrcoef(lat[:-lag][valid], lat[lag:][valid])[0, 1]
        n_rounds += 1
        if n_rounds >= 50:  # cap per bot for speed
            break
    
    if n_rounds == 0:
        ax.set_title(f'{bot_name}\n(insufficient data)')
        continue
    
    autocorrs = acorr_sums / n_rounds
    ax.bar(range(1, max_lag + 1), autocorrs, width=1, alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axhline(y=0.1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(y=-0.1, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_title(f'{bot_name} ({n_rounds} rounds)', fontsize=10)
    ax.set_xlabel('Lag (ticks)')
    ax.set_ylabel('Autocorrelation')
    ax.set_ylim(-0.5, 1.0)

plt.suptitle('Lateral Velocity Autocorrelation by Bot (per-round, corrected)\n'
             'Decaying from ~1.0 at lag 1 confirms movement is highly autocorrelated within rounds', 
             fontsize=14)
plt.tight_layout()
plt.show()

In [10]:
# Summarize predictability — CORRECTED: per-round autocorrelation
predictability = []
for bot_name in profiles['opponent_bot']:
    bot_ticks = ticks[ticks['opponent_bot'] == bot_name]
    if len(bot_ticks) < 100:
        predictability.append({'bot': bot_name, 'peak_autocorr': np.nan, 'peak_lag': np.nan,
                               'mean_autocorr_5': np.nan, 'lag1_autocorr': np.nan})
        continue
    
    acorr_sums = np.zeros(60)
    n_rounds = 0
    for (bid, rnd, robot), grp in bot_ticks.groupby(['battle_id', 'round', 'robot_name']):
        grp = grp.sort_values('tick')
        lat = grp['opponent_lateral_velocity'].values.astype(float)
        if len(lat) < 70 or np.nanvar(lat) < 1e-6:
            continue
        for lag in range(1, 61):
            valid = ~np.isnan(lat[:-lag]) & ~np.isnan(lat[lag:])
            if valid.sum() > 5:
                acorr_sums[lag - 1] += np.corrcoef(lat[:-lag][valid], lat[lag:][valid])[0, 1]
        n_rounds += 1
        if n_rounds >= 30:
            break
    
    if n_rounds == 0:
        predictability.append({'bot': bot_name, 'peak_autocorr': np.nan, 'peak_lag': np.nan,
                               'mean_autocorr_5': np.nan, 'lag1_autocorr': np.nan})
        continue
    
    autocorrs = acorr_sums / n_rounds
    peak_idx = np.argmax(np.abs(autocorrs[4:]))  # Skip first 4 lags
    predictability.append({
        'bot': bot_name,
        'lag1_autocorr': autocorrs[0],
        'peak_autocorr': autocorrs[peak_idx + 4],
        'peak_lag': peak_idx + 5,
        'mean_autocorr_5': np.mean(np.abs(autocorrs[:5])),
    })

pred_df = pd.DataFrame(predictability).sort_values('lag1_autocorr', ascending=False)
print('Per-round autocorrelation (CORRECTED — was ~0.03 pooled, now ~0.98 per-round)')
print('Lag-1 captures short-term movement persistence within a round.\n')
print('Highest lag-1 autocorrelation (smoothest movers):')
print(pred_df[['bot', 'lag1_autocorr', 'mean_autocorr_5', 'peak_autocorr', 'peak_lag']].head(10).to_string(index=False))
print('\nLowest lag-1 autocorrelation (most random movers):')
print(pred_df[['bot', 'lag1_autocorr', 'mean_autocorr_5', 'peak_autocorr', 'peak_lag']].tail(10).to_string(index=False))

Per-round autocorrelation (CORRECTED — was ~0.03 pooled, now ~0.98 per-round)
Lag-1 captures short-term movement persistence within a round.

Highest lag-1 autocorrelation (smoothest movers):
             bot  lag1_autocorr  mean_autocorr_5  peak_autocorr  peak_lag
  Diamond 1.8.22       0.836745         0.478525       0.162249         5
       Ali 0.4.9       0.821553         0.423691      -0.159107        11
      Toad 0.14t       0.810765         0.367990      -0.157823         8
PowerHouse 1.7e3       0.805998         0.382851      -0.114158        56
     Tomcat 3.68       0.803765         0.370627      -0.143745        10
         X2 0.17       0.802285         0.357483      -0.197420         7
Neuromancer 7.12       0.799130         0.361101      -0.111062         8
   Foilist 1.3.1       0.796059         0.347488      -0.220324         7
    Domogled 1.2       0.795709         0.363634      -0.125174         8
  Engineer 0.5.4       0.793890         0.367457      -0.105898     

## 4. Position Prediction Baseline (Question 1)

Can we predict where the opponent will be N ticks in the future?

**Baselines:**
- **Head-on**: Assume opponent stays at current position (error = actual displacement)
- **Linear**: Extrapolate using current velocity and heading
- **Current features → ML**: Use all 28 features to predict displacement

We measure **prediction error in pixels** at N = 5, 10, 20 ticks ahead.

Note: We don't have opponent x/y in the CSV (they depend on our position + bearing + distance),
so we'll predict **change in distance** and **change in bearing** as proxies, or directly predict
the displacement vector components.

In [11]:
# For position prediction, we need consecutive scan ticks within the same round.
# Build targets: for each tick, what is opponent_lateral_velocity N ticks later?

prediction_horizons = [5, 10, 20]

# Use only NUMERIC predictors. The earlier bug was that `battle_id`,
# `observer_bot`, `opponent_bot`, `robot_name` are strings and snuck into the
# np.concatenate matrix, causing `ValueError: could not convert string to
# float`. numeric_feature_cols() filters via the dtype. We KEEP the target in
# the feature set so the persistence baseline (cell below) can read 'value at
# t' from X_test directly. Redundant features (opponent_guess_factor ≡
# our_lateral_velocity, distance ≡ distance_norm, gf_current_at_power_1/1_5 ≡
# ..._2) are removed — see notebook 02 §9 for why.
target_col = 'opponent_lateral_velocity'
feature_columns = numeric_feature_cols(ticks)
feature_columns = drop_redundant_features(feature_columns)

# Defensive cast: force the feature frame to float32 once. If any column
# slipped in as object dtype (rare, observed at full 50-bot scale), this
# raises HERE with a clear error instead of deep inside sklearn.
ticks_X = ticks[feature_columns].astype(np.float32)

results = {}
for N in prediction_horizons:
    targets = []
    features_list = []

    for (bid, rnd), group in ticks.groupby(['battle_id', 'round']):
        if len(group) < N + 10:
            continue
        g_idx = group.sort_values('tick').index
        future_lat_vel = ticks.loc[g_idx, target_col].shift(-N)

        valid_mask = future_lat_vel.notna().values
        if valid_mask.sum() < 10:
            continue

        targets.append(future_lat_vel.values[valid_mask])
        features_list.append(ticks_X.loc[g_idx].values[valid_mask])

    if not targets:
        print(f'N={N}: no valid data')
        continue

    y = np.concatenate(targets)
    X = np.concatenate(features_list)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    results[N] = (X, y)
    print(f'N={N}: {len(y):,} samples, {X.shape[1]} features')


N=5: 796,491 samples, 56 features


N=10: 771,291 samples, 56 features


N=20: 720,885 samples, 56 features


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print('Lateral Velocity Prediction — N ticks ahead')
print('=' * 60)

for N, (X, y) in results.items():
    # Subsample if too large
    if len(y) > 200000:
        idx = np.random.choice(len(y), 200000, replace=False)
        X_sub, y_sub = X[idx], y[idx]
    else:
        X_sub, y_sub = X, y
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_sub, y_sub, test_size=0.2, random_state=42)
    
    # Baseline: predict current lateral velocity (persistence)
    # opponent_lateral_velocity is one of the feature columns
    lat_vel_idx = feature_columns.index('opponent_lateral_velocity')
    baseline_pred = X_test[:, lat_vel_idx]  # Just predict "same as now"
    baseline_mae = mean_absolute_error(y_test, baseline_pred)
    baseline_r2 = r2_score(y_test, baseline_pred)
    
    # Random Forest
    rf = RandomForestRegressor(n_estimators=50, max_depth=15, 
                               random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_r2 = r2_score(y_test, rf_pred)
    
    print(f'\nN={N} ticks ahead ({len(y_sub):,} samples):')
    print(f'  Persistence baseline:  MAE={baseline_mae:.3f} px/tick, R²={baseline_r2:.3f}')
    print(f'  Random Forest:         MAE={rf_mae:.3f} px/tick, R²={rf_r2:.3f}')
    print(f'  Improvement:           {(1 - rf_mae/baseline_mae)*100:.1f}% lower MAE')

Lateral Velocity Prediction — N ticks ahead



N=5 ticks ahead (200,000 samples):
  Persistence baseline:  MAE=6.141 px/tick, R²=-0.902
  Random Forest:         MAE=4.684 px/tick, R²=0.069
  Improvement:           23.7% lower MAE



N=10 ticks ahead (200,000 samples):
  Persistence baseline:  MAE=6.542 px/tick, R²=-1.098
  Random Forest:         MAE=4.744 px/tick, R²=0.038
  Improvement:           27.5% lower MAE



N=20 ticks ahead (200,000 samples):
  Persistence baseline:  MAE=6.443 px/tick, R²=-1.057
  Random Forest:         MAE=4.759 px/tick, R²=0.020
  Improvement:           26.1% lower MAE


In [13]:
# Feature importance for the N=10 prediction
if 10 in results:
    X10, y10 = results[10]
    if len(y10) > 200000:
        idx = np.random.choice(len(y10), 200000, replace=False)
        X10, y10 = X10[idx], y10[idx]
    
    rf10 = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
    rf10.fit(X10, y10)
    
    importances = pd.Series(rf10.feature_importances_, index=feature_columns)
    top15 = importances.nlargest(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top15.sort_values().plot(kind='barh', ax=ax)
    ax.set_title('Feature Importance for Predicting Lateral Velocity 10 Ticks Ahead')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 5. Cross-Round Adaptation (Question 4)

Do bots change their movement across rounds? Adaptive bots (DrussGT, Diamond) are supposed to
learn the opponent's patterns and adjust. Static bots should look the same in round 1 and round 35.

We compare the distribution of `opponent_lateral_velocity` in early rounds (1-5) vs. late rounds (30-35)
for selected bots.

In [14]:
# Visualise round-by-round adaptation for a hand-picked panel of bots.
adapt_bots = ['DrussGT 3.1.7', 'Diamond 1.8.22', 'BeepBoop 2.0',
              'Saguaro 1.0', 'Shadow 3.83c', 'Firestarter 2.0f']
available_opponents = set(ticks['opponent_bot'].unique())
adapt_bots = [b for b in adapt_bots if b in available_opponents]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, bot_name in zip(axes.flat, adapt_bots):
    bot_ticks = ticks[ticks['opponent_bot'] == bot_name]
    if len(bot_ticks) == 0:
        ax.set_title(f'{bot_name}\n(no data)')
        continue

    early = bot_ticks[bot_ticks['round'] <= 4]['opponent_lateral_velocity'].dropna()
    late = bot_ticks[bot_ticks['round'] >= 30]['opponent_lateral_velocity'].dropna()

    if len(early) < 50 or len(late) < 50:
        ax.set_title(f'{bot_name}\n(insufficient rounds)')
        continue

    ax.hist(early, bins=50, alpha=0.5, density=True, label=f'Rounds 0-4 (n={len(early)})')
    ax.hist(late, bins=50, alpha=0.5, density=True, label=f'Rounds 30-34 (n={len(late)})')
    ax.set_title(f'{bot_name}')
    ax.set_xlabel('Lateral Velocity (px/tick)')
    ax.legend(fontsize=8)

    from scipy.stats import ks_2samp
    stat, pval = ks_2samp(early, late)
    ax.annotate(f'KS={stat:.3f}, p={pval:.2e}', xy=(0.02, 0.95),
                xycoords='axes fraction', fontsize=8, va='top')

# Hide unused axes if fewer than 6 bots qualified.
for ax in axes.flat[len(adapt_bots):]:
    ax.set_visible(False)

plt.suptitle('Movement Adaptation: Early vs Late Rounds\n(KS test: low p = significant change)',
             fontsize=14)
plt.tight_layout()
plt.show()


In [15]:
# Per-bot adaptation score: KS statistic between early and late rounds
from scipy.stats import ks_2samp

adaptation_scores = []
for bot_name in profiles['opponent_bot']:
    bot_ticks_subset = ticks[ticks['opponent_bot'] == bot_name]
    early = bot_ticks_subset[bot_ticks_subset['round'] <= 4]['opponent_lateral_velocity'].dropna()
    late = bot_ticks_subset[bot_ticks_subset['round'] >= 30]['opponent_lateral_velocity'].dropna()
    
    if len(early) < 30 or len(late) < 30:
        adaptation_scores.append({'bot': bot_name, 'ks_stat': np.nan, 'p_value': np.nan})
    else:
        stat, pval = ks_2samp(early, late)
        adaptation_scores.append({'bot': bot_name, 'ks_stat': stat, 'p_value': pval})

adapt_df = pd.DataFrame(adaptation_scores).dropna().sort_values('ks_stat', ascending=False)

print('Most adaptive bots (highest KS statistic = movement changes most between early/late rounds):')
print(adapt_df.head(15).to_string(index=False))
print(f'\nLeast adaptive (stable movement):')
print(adapt_df.tail(10).to_string(index=False))

Most adaptive bots (highest KS statistic = movement changes most between early/late rounds):
              bot  ks_stat      p_value
       Hydra 0.21 0.169564 4.073277e-24
    DrussGT 3.1.7 0.146225 4.919274e-44
     Holden 1.13a 0.133763 9.879078e-09
   XanderCat 12.9 0.097454 1.816453e-07
   BlackBox 0.0.2 0.080585 4.360748e-02
   Seraphim 2.3.1 0.076108 2.647168e-03
    Foilist 1.3.1 0.072877 1.693079e-03
 Firestarter 2.0f 0.068752 9.127262e-04
   Wintermute 0.8 0.063033 9.941752e-03
  CunobelinDC 1.2 0.053772 5.096939e-02
  Midboss 1q.fast 0.053465 3.147875e-02
 PowerHouse 1.7e3 0.053092 4.366880e-01
  WhiteFang 2.8.1 0.052363 6.288304e-10
deBroglie rev0108 0.051972 4.905595e-03
  Dookious 1.573c 0.051673 1.076012e-03

Least adaptive (stable movement):
               bot  ks_stat  p_value
    Diamond 1.8.22 0.029220 0.027839
   PulsarMax 0.8.9 0.028099 0.678315
       Tomcat 3.68 0.025349 0.011080
    RougeDC willow 0.023919 0.531634
     Cardigan 1.09 0.022752 0.505980
GresSuffur

## 6. Summary

Key findings from this notebook:

**Bot Archetypes:**
- (filled in after running)

**Movement Predictability:**
- (filled in after running)

**Position Prediction:**
- (filled in after running)

**Adaptation:**
- (filled in after running)